## Install & Import

In [ ]:
import pandas as pd
import numpy as np
import json
import joblib
import os
import matplotlib.pyplot as plt
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.feature_selection import RFECV
from sklearn.model_selection import KFold
from snowflake.snowpark.context import get_active_session

session = get_active_session()
session.sql("USE DATABASE SNOWFLAKE_LEARNING_DB").collect()
session.sql("USE SCHEMA PUBLIC").collect()
STAGE = "@SNOWFLAKE_LEARNING_DB.PUBLIC.EXTERNAL_DATA_STAGE"

TARGET_COLS = [
    'Electrical Conductance',
    'Total Alkalinity',
    'Dissolved Reactive Phosphorus'
]
TARGET_SHORT = {
    'Electrical Conductance': 'ec',
    'Total Alkalinity': 'alkalinity',
    'Dissolved Reactive Phosphorus': 'drp'
}

# Load master data
local = "/tmp/master_train.parquet"
if not os.path.exists(local):
    session.file.get(f"{STAGE}/master_train.parquet", "/tmp/")
master_train = pd.read_parquet(local)

# Load metadata
meta_local = "/tmp/pipeline_meta.json"
if not os.path.exists(meta_local):
    session.file.get(f"{STAGE}/pipeline_meta.json", "/tmp/")
with open(meta_local) as f:
    meta = json.load(f)

feature_cols = [c for c in master_train.columns if c not in TARGET_COLS and c != 'Region']
X = master_train[feature_cols]
region_series = master_train['Region'].copy()
region_series.fillna('Unknown', inplace=True)
region_array, region_uniques = pd.factorize(region_series)

print(f"✓ Loaded: {X.shape[0]} rows × {X.shape[1]} features")
print(f"  Targets: {TARGET_COLS}")
print(f"  Regions encoded: {len(region_uniques)} unique")

In [ ]:
master_train.info()

In [ ]:
region_series = master_train['Region'].copy()

if region_series.isna().any():
    print(f"Terdapat {region_series.isna().sum()} nilai REGION yang hilang. Mengisi dengan 'Unknown'.")
    region_series.fillna('Unknown', inplace=True)

region_encoded, region_uniques = pd.factorize(region_series)
print(f"REGION telah di-encode menjadi {len(region_uniques)} kategori unik:")
for i, reg in enumerate(region_uniques):
    print(f"  {i}: {reg}")

region_array = region_encoded
region_mapping = dict(enumerate(region_uniques))
# Misal simpan sebagai JSON
# with open('/tmp/region_mapping.json', 'w') as f:
#     json.dump(region_mapping, f)

In [ ]:
master_train

In [ ]:
from sklearn.model_selection import GroupKFold

# region_bundle = joblib.load('/tmp/region_groups.joblib')
region_array = region_bundle['region_array'] 
print(f"Regions loaded: {region_bundle['region_counts']}")

def run_rfecv_for_target(X, y, groups, target_name, n_splits=4, step=1, 
                          n_estimators=100, random_state=42):
    """
    RFECV dengan GroupKFold — setiap fold = 1 region held-out.
    n_splits = jumlah region (4: Western_Cape, Northern_Bulk, Eastern_Cape, Karoo_Buffer)
    """
    valid_mask = y.notna()
    X_valid = X[valid_mask].copy()
    y_valid = y[valid_mask].copy()
    groups_valid = groups[valid_mask]
    
    skew = y_valid.skew()
    if abs(skew) > 1:
        y_valid = np.log1p(y_valid)
    
    estimator = ExtraTreesRegressor(
        n_estimators=n_estimators, max_depth=None,
        random_state=random_state, n_jobs=-1
    )
    
    cv = GroupKFold(n_splits=n_splits)
    
    rfecv = RFECV(
        estimator=estimator, step=step,
        cv=cv.split(X_valid, y_valid, groups_valid), 
        scoring='r2', min_features_to_select=5, n_jobs=-1
    )
    rfecv.fit(X_valid, y_valid)
    
    selected = X_valid.columns[rfecv.support_].tolist()
    print(f"  ✓ Optimal: {rfecv.n_features_} features, Best R²: "
          f"{rfecv.cv_results_['mean_test_score'].max():.4f}")
    return rfecv, selected

def plot_rfecv_curve(rfecv, target_name):
    """Plot jumlah fitur vs CV R² score."""
    scores = rfecv.cv_results_['mean_test_score']
    stds = rfecv.cv_results_['std_test_score']
    n_features_range = range(rfecv.min_features_to_select, 
                              rfecv.min_features_to_select + len(scores))
    
    plt.figure(figsize=(10, 5))
    plt.plot(n_features_range, scores, 'b-', linewidth=2)
    plt.fill_between(n_features_range, 
                     np.array(scores) - np.array(stds),
                     np.array(scores) + np.array(stds), alpha=0.2)
    plt.axvline(rfecv.n_features_, color='red', linestyle='--', 
                label=f'Optimal: {rfecv.n_features_} features (R²={scores[rfecv.n_features_ - rfecv.min_features_to_select]:.4f})')
    plt.xlabel('Number of Features')
    plt.ylabel('CV R² Score')
    plt.title(f'RFECV — {target_name}')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

print(" RFECV engine ready")

In [ ]:
all_selected_features = {}
all_rfecv_objects = {}

for target_col in TARGET_COLS:
    short = TARGET_SHORT[target_col]
    y = master_train[target_col]
    
    rfecv_obj, selected = run_rfecv_for_target(
    X=X, 
    y=y, 
    groups=region_array, 
    target_name=short, 
    n_splits=4 )
    
    all_selected_features[target_col] = selected
    all_rfecv_objects[short] = rfecv_obj
    plot_rfecv_curve(rfecv_obj, short)

# Summary
print("\n" + "=" * 60)
print("RFECV SUMMARY")
print("=" * 60)
for target_col, feats in all_selected_features.items():
    short = TARGET_SHORT[target_col]
    print(f"  {short}: {len(feats)} features selected")

# Fitur yang shared antar 3 target (sangat penting!)
shared = set(all_selected_features[TARGET_COLS[0]])
for tc in TARGET_COLS[1:]:
    shared &= set(all_selected_features[tc])
print(f"\n  Shared across ALL 3 targets: {len(shared)} features")
if shared:
    print(f"  → {sorted(shared)}")

In [ ]:
output = {
    'rfecv_config': {
        'n_splits': 5, 'step': 1, 'n_estimators': 100,
        'min_features_to_select': 5
    },
    'selected_features': {},
    'n_original_features': X.shape[1],
}

for target_col, feats in all_selected_features.items():
    short = TARGET_SHORT[target_col]
    best_idx = all_rfecv_objects[short].n_features_ - all_rfecv_objects[short].min_features_to_select
    best_score = float(all_rfecv_objects[short].cv_results_['mean_test_score'][best_idx])
    output['selected_features'][short] = {
        'target_column': target_col,
        'features': feats,
        'n_selected': len(feats),
        'best_cv_r2': round(best_score, 4)
    }

with open('/tmp/selected_features.json', 'w') as f:
    json.dump(output, f, indent=2)
session.file.put("file:///tmp/selected_features.json", STAGE, auto_compress=False, overwrite=True)

print("selected_features.json saved to Stage")
for short, info in output['selected_features'].items():
    print(f"  {short}: {info['n_selected']} features, CV R²={info['best_cv_r2']}")